# OIBSIP – Level 1 Task 2: Customer Segmentation Analysis

**Dataset:** UCI Machine Learning Repository – Online Retail (genuine historical transaction data)  
**Method:** RFM analysis + K-Means clustering  
**Tools:** Python, pandas, scikit-learn, matplotlib

## Objective
Group customers by purchasing behaviour to support targeted marketing. This project uses Recency, Frequency and Monetary value (RFM). Customer Lifetime Value is represented as observed-period revenue, not a predictive lifetime value.

**Data note:** The source is historical (1 Dec 2010–9 Dec 2011). Rows without CustomerID cannot be assigned to an individual customer and are excluded from customer segmentation. Revenue is quantity × unit price, not profit.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Put "Online Retail.xlsx" in the notebook's working folder, or upload it when prompted.
excel_path = Path("Online Retail.xlsx")
if not excel_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        excel_path = Path(next(iter(uploaded.keys())))
    except Exception:
        raise FileNotFoundError("Place the UCI source file 'Online Retail.xlsx' beside this notebook.")

tx = pd.read_excel(excel_path)
tx["InvoiceNo"] = tx["InvoiceNo"].astype(str)
tx["InvoiceDate"] = pd.to_datetime(tx["InvoiceDate"], errors="coerce")
tx["CustomerID"] = pd.to_numeric(tx["CustomerID"], errors="coerce")
tx["UnitPrice"] = pd.to_numeric(tx["UnitPrice"], errors="coerce")
tx["Quantity"] = pd.to_numeric(tx["Quantity"], errors="coerce")
tx = tx.dropna(subset=["CustomerID", "InvoiceDate", "InvoiceNo", "UnitPrice", "Quantity"])
tx = tx[~tx["InvoiceNo"].str.startswith("C", na=False)]
tx = tx[(tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)]
tx["Revenue"] = tx["Quantity"] * tx["UnitPrice"]
tx["CustomerID"] = tx["CustomerID"].astype(int).astype(str)
print("Clean transaction rows:", len(tx))
print("Customers available for segmentation:", tx["CustomerID"].nunique())
display(tx.head())


## 1. Build customer-level RFM features

- **Recency:** days since the customer's most recent purchase (lower is better).
- **Frequency:** number of distinct invoices (higher usually means more engaged).
- **Monetary:** total observed revenue from the customer (higher means greater observed value).
- **Average Order Value:** Monetary ÷ Frequency.
- **Observed customer span:** days between first and last recorded purchase; this is descriptive, not true lifetime.


In [ ]:
snapshot_date = tx["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
rfm = tx.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max().normalize()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
    First_Purchase=("InvoiceDate", "min"),
    Last_Purchase=("InvoiceDate", "max"),
    Country=("Country", lambda x: x.mode().iat[0] if not x.mode().empty else "Unknown")
).reset_index()
rfm["Average_Order_Value"] = rfm["Monetary"] / rfm["Frequency"]
rfm["Customer_Lifetime_Days"] = (rfm["Last_Purchase"] - rfm["First_Purchase"]).dt.days + 1
display(rfm[["Recency","Frequency","Monetary","Average_Order_Value","Customer_Lifetime_Days"]].describe().T)


## 2. Prepare features and standardise\nRFM values are highly skewed, so apply `log1p` before StandardScaler. This reduces the influence of extreme spenders while retaining customer ordering.

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
X_log = np.log1p(rfm[features])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)


## 3. Elbow Method and K-Means\nCompare inertia for K=2 to K=8. K=4 is used here as an interpretable four-segment business solution; the elbow curve should also be reviewed rather than treating K=4 as a universal optimum.

In [ ]:
inertias = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2,9), inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.tight_layout()
plt.show()

kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)
print("Silhouette score (K=4):", round(silhouette_score(X_scaled, rfm["Cluster"]), 3))


## 4. Cluster profiles and segment names\nClusters are named by observed average Monetary value. Labels are descriptive marketing personas, not demographic identities.

In [ ]:
profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID","count"),
    Avg_Recency_Days=("Recency","mean"),
    Avg_Purchase_Frequency=("Frequency","mean"),
    Avg_Monetary=("Monetary","mean"),
    Avg_Order_Value=("Average_Order_Value","mean"),
    Avg_Lifetime_Days=("Customer_Lifetime_Days","mean")
).reset_index()
order = profile.sort_values("Avg_Monetary", ascending=False)["Cluster"].tolist()
segment_names = ["Champions / High Value", "Loyal Regulars", "Potential / Growing", "At-Risk / Low Engagement"]
mapping = {cluster: segment_names[i] for i, cluster in enumerate(order)}
rfm["Segment"] = rfm["Cluster"].map(mapping)
profile["Segment"] = profile["Cluster"].map(mapping)
display(profile.sort_values("Avg_Monetary", ascending=False))


## 5. Visualise customer segments

In [ ]:
plt.figure(figsize=(8,5))
for segment, group in rfm.groupby("Segment"):
    plt.scatter(group["Frequency"], group["Monetary"], alpha=.55, s=18, label=segment)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Purchase frequency (log scale)"); plt.ylabel("Monetary value (£, log scale)")
plt.title("Frequency vs Monetary by Customer Segment"); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
for segment, group in rfm.groupby("Segment"):
    plt.scatter(group["Recency"], group["Monetary"], alpha=.55, s=18, label=segment)
plt.yscale("log")
plt.xlabel("Recency (days)"); plt.ylabel("Monetary value (£, log scale)")
plt.title("Recency vs Monetary by Customer Segment"); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

rfm["Segment"].value_counts().reindex(segment_names).plot(kind="bar", figsize=(8,5))
plt.ylabel("Number of customers"); plt.title("Customers per Segment"); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()


## 6. Marketing recommendations

- **Champions / High Value:** VIP rewards, early access, referral benefits and proactive service.
- **Loyal Regulars:** loyalty milestones, replenishment reminders and subscription/bundle tests.
- **Potential / Growing:** relevant cross-sells and small, controlled incentives to encourage repeat purchase.
- **At-Risk / Low Engagement:** targeted win-back campaigns and product recommendations; avoid blanket discounts.

## Limitations
- CustomerID-missing transactions cannot be linked to a customer and are excluded.
- Dataset covers a limited historical period; segment labels describe behaviour in that period only.
- Monetary is observed revenue, not profit or predictive lifetime value.
- K-Means is sensitive to feature choices and K; compare the elbow curve and silhouette score before operational use.


In [ ]:
# Optional: export reusable results
Path("outputs").mkdir(exist_ok=True)
rfm.to_csv("outputs/customer_rfm_segments.csv", index=False)
profile.to_csv("outputs/cluster_profiles.csv", index=False)
pd.DataFrame({"k": range(2,9), "inertia": inertias}).to_csv("outputs/elbow_inertia.csv", index=False)
print("Saved RFM segments and cluster profiles in outputs/.")
